Imports

In [56]:
import geocoder
import chromadb
import uuid
import time
import requests
import webbrowser
import sys
from sentence_transformers import SentenceTransformer

embedding model laden

In [57]:
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3857.18it/s]


vector Database aanmaken

In [58]:
client = chromadb.PersistentClient(path="./recept_database")

collection = client.get_or_create_collection(
    name="recepten"
)

gekozen ingredienten invullen

In [60]:
items = []

print('')

while True:
    item = input("Welke ingredienten zou je willen gebruiken? Het mogen ingredienten van huis zijn of die je nog moet halen in de supermarkt(type stop om te stoppen): ")

    if item == "stop":
        break

    items.append(item)

selecteren welke supermarkt je boodschappen doet

In [61]:
destination = input('wat is de supermarkt waar je heen wilt( vul supermarkt in of niks)')

In [ ]:
destination = destination.lower()

if  "albert heijn" in destination or "alber" in destination or 'hein' in destination or destination == "ah":
    supermarkt = "https://www.ah.nl/zoeken?query="

elif "plus" in destination:
    supermarkt = "https://www.plus.nl/zoekresultaten?SearchTerm="

elif "jumbo" in destination or "jum" in destination:
    supermarkt = "https://www.jumbo.com/producten/?searchType=keyword&searchTerms="

elif "spar" in destination or 'spa' in destination:
    supermarkt = "https://www.spar.nl/zoek/?fq="

elif "lidl" in destination or "lid" in destination or "lit" in destination or "lil" in destination:
    supermarkt = 'https://www.lidl.nl/q/search?q='

else:
    supermarkt = "https://google.com/search?q=supermarkt+voor+"

???

In [63]:
g = geocoder.ip("me")
regio = g.city
lat = g.latlng[0]
lon = g.latlng[1]

Query voor de vector DB

In [64]:
def build_query(items):
    return f"""
Ik wil koken met: {', '.join(items)}.
Ik zoek gerechten die hierbij passen.
"""

Uitzoeken welke rcepten erop lijken

In [65]:
def get_similar_recipes(items, amount=5):

    query = build_query(items)
    embedding = embedding_model.encode(query)

    result = collection.query(
        query_embeddings=[embedding.tolist()],
        n_results=amount,
        where={
            "liked": True
        }
    )

    if len(result["documents"]) == 0:
        return result

    if len(result["documents"][0]) == 0:
        return result

    return result

Formatteren voor betere prompting

In [66]:
def format_similar(results):
    if not results["documents"]:
        return ""

    return "\n".join(results["documents"][0][:5])

Verglijkbare waardes in een var stoppen

In [67]:
print(lat)
print(lon)

results = get_similar_recipes(items)
similar_recipes = format_similar(results)

51.5317
4.2208


De standaard prompt

In [68]:
prompts = [
    f"""
Genereer EXACT 5 verschillende gerechten op basis van deze ingrediënten:

{items}

Vergelijkbare recepten uit eerdere sessies:

{similar_recipes}

BELANGRIJKE REGELS:
- Geef EXACT 5 regels terug.
- Geen uitleg voor of na de regels.
- Geen markdown.
- Geen opsommingstekens.
- Elke regel moet EXACT dit formaat hebben:

gerecht_nr | gerecht | ingredienten | duur | nieuw ingredient = (ingredient)

VOORBEELD:
1 | Pasta met kip | kip, pasta, ui | 25 min | nieuw ingredient = (parmezaanse kaas)

EXTRA REGELS:
- Gebruik zoveel mogelijk ingrediënten uit de opgegeven lijst.
- Maximaal 1 nieuw ingrediënt per gerecht.
- Bereidingstijd maximaal 30 minuten.
- Het nieuwe ingrediënt MOET tussen haakjes staan.
- De tekst 'nieuw ingredient = ...' moet altijd het laatste onderdeel van de regel zijn.
- Gebruik het pipe-teken '|' uitsluitend als scheidingsteken.
- Gebruik geen pipe-tekens in gerechtnamen of ingrediënten.
- Nummer de gerechten van 1 t/m 5.

Geef alleen de 5 regels terug.
"""
]

Recepten opslaan

In [69]:
def save_recipe(recipe_text):

    embedding = embedding_model.encode(recipe_text)

    collection.add(
    ids=[str(uuid.uuid4())],
    documents=[recipe_text],
    embeddings=[embedding.tolist()],
    metadatas=[{
        "ingredients" : ",".join(items),
        "liked": True
    }]
)

Tijd meten starten

In [70]:
for prompt in prompts:
    start_time = time.time()

MetaData opslaan

In [71]:
data = {
    "model": "nvidia/nemotron-3-nano-4b",
    "messages": [
        {"role": "user", "content": prompt}
    ],
    "temperature": 0.7
}


Bot functie

In [72]:
url = 'http://127.0.0.1:1234/v1/chat/completions'



def lmbot():
    try:
        response = requests.post(url, json=data)
        response.raise_for_status()

        end_time = time.time()

        result = response.json()

        answer = result["choices"][0]["message"]["content"]
        model_name = result.get("model", "onbekend")
        response_length = len(answer)

    except requests.exceptions.RequestException as e:
        print(f"Fout bij API-aanroep: {e}")

    regels = []

    for regel in answer.splitlines():
        if "nieuw ingredient" in regel.lower():
            ingredient = regel.split("=", 1)[1].strip()
            ingredient = ingredient.strip("()[]{}")
            ingredient = ingredient.replace(" ", "+")

            url2 = supermarkt + ingredient
            regel += f"|{url2}"

        regels.append(regel)

    return "\n".join(regels)

uitvoeren van de eerste stap van de agent

In [73]:
while True:
    bot = lmbot()

    print(bot)
    
    check = input("Zit hier een goed gerecht tussen? (Ja/Nee)")

    if check.lower() == "ja":
        break



1 | Pasta met spek en ui | pasta, speck, ui | 20 min | nieuw ingredient = (mozzarella)|https://google.com/search?q=supermarkt+voor+mozzarella
2 | Uiworstkaas | ui, worst, kaas | 20 min | nieuw ingredient = (parmezaanse kaas)|https://google.com/search?q=supermarkt+voor+parmezaanse+kaas
3 | Spekpasta met ui | pasta, speck, ui | 15 min | nieuw ingredient = (mozzarella)|https://google.com/search?q=supermarkt+voor+mozzarella
4 | Pasta met kaas en ui | pasta, kaas, ui | 20 min | nieuw ingredient = (mozzarella)|https://google.com/search?q=supermarkt+voor+mozzarella
5 | Spekkaasui | speck, kaas, ui | 10 min | nieuw ingredient = (parmezaanse kaas)|https://google.com/search?q=supermarkt+voor+parmezaanse+kaas

1 | Pasta met sperk en ui | kaas, ui, pasta, sperk | 20 min | nieuw ingrediënt = (mozzarella)  
2 | Spekui ui kaas pasta | ui, kaas, pasta, sperk | 15 min | nieuw ingrediënt = (parmezaanse kaas)  
3 | Kaas Ui Spek Sup | ui, kaas | 10 min | nieuw ingrediënt = (provolone)  
4 | Ui Spek Kaas 

Hele geselecteerde recept uitprinten

In [74]:
recept_nummer = int(input("Welk nummer van recept wil je hebben? "))

for regel in bot.splitlines():
    if regel.startswith(f"{recept_nummer} |"):
        recept_select = regel
        break
save_recipe(recept_select)

prompts = [
    f"""
    maak een recept voor:{recept_select}
    """
]

for prompt in prompts:
    start_time = time.time()

data = {
    "model": "nvidia/nemotron-3-nano-4b",
    "messages": [
        {"role": "user", "content": prompt}
    ],
    "temperature": 0.7
}

bot = lmbot()
print(bot)

full_recipe = bot
save_recipe(full_recipe)


**Pasta met sperk, ui & mozzarella – 20 minuten**

---

### Ingrediënten (voor 4 personen)

| Hoeveelheid | Ingrediënt |
|-------------|-----------|
| 320 g | spaghetti (of je liever een snellere pasta) |
| 1 | grote ui, fijngehakt |
| 80 ml | olijfolie (plus extra voor de mozzarella) |
| ½ tl | poederspier in witte zout |
| 2 knoopjes | verse of gedroogde roodpeper, fijngehakt |
| 150 g | schaalfijne mozzarella (in blokjes) |
| Eventueel | een snufje geraspte peterselie of basilicum voor garnish |

---

### Voorbereiding (≈ 20 minuten)

1. **Water koken**  
   - Breng 8 l water aan de kook, voeg zout en poederspier in.  
   - Gebruik een snelle kokkendeur of een pannenkoekenplaat (alleen voor het opwarmen van de mozzarella).

2. **Ui & sperk**  
   - Verwijder de ui uit de schil en haag fijn.  
   - Meng met 80 ml olijfolie, een snufje zout en poederspier in een grote pan.  
   - Kook op laag vuur 5‑7 minuten tot de ui knapperig is en eventueel licht bruin.

3. **Pasta koken**  
   -

Supermarkt route

In [ ]:
boy = input('Wil je een kaart naar de supermarkt in de buurt (ja/nee)')

if boy.lower() == 'ja':
    if destination in ("niks", ""):
        url2 = (
            f"https://www.google.com/maps/dir/?api=1"
            f"&origin={lat},{lon}"
            f"&destination=supermarkt"
            f"&travelmode=driving"
        )
        webbrowser.open(url2)

    else:
        url2 = (
            f"https://www.google.com/maps/dir/?api=1"
            f"&origin={lat},{lon}"
            f"&destination={destination}"
            f"&travelmode=driving"
        )

        webbrowser.open(url2)
else:
    exit()

SystemExit: 

C:\Users\Axis\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
